# Dataset overview

Summarize schemas, loaded-table completeness, duplicate rows, dates and numeric distributions. Unscanned source tables are labeled as such rather than assigned estimated statistics.

In [ ]:
from pathlib import Path
import sys

ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents)
            if (path / 'backend' / 'app').is_dir())
sys.path.insert(0, str(ROOT)) if str(ROOT) not in sys.path else None
import pandas as pd
import plotly.express as px
from backend.app import eda

DATA_DIRS = [ROOT / 'data' / 'processed', ROOT / 'data' / 'raw', ROOT / 'Raw']
overview = eda.overview(DATA_DIRS)
overview['counts'], overview['date_coverage']

In [ ]:
display(pd.DataFrame(overview['datasets']))
table_quality = pd.DataFrame.from_dict(overview['tables'], orient='index')
display(table_quality[['available', 'rows', 'columns', 'missing_cells', 'completeness_percent', 'duplicate_rows']])

In [ ]:
missing_rows = []
for table, values in overview['tables'].items():
    if values.get('available'):
        missing_rows.extend({'table': table, 'column': name, 'missing': count}
                            for name, count in values['missing_values'].items())
if missing_rows:
    fig = px.bar(pd.DataFrame(missing_rows).sort_values('missing', ascending=False).head(30),
                 x='column', y='missing', color='table', title='Columns with most missing values')
    fig.show()
else:
    print('No missing values in the loaded tables, or no analysis tables were available.')

In [ ]:
games = eda.games_analysis(DATA_DIRS)
if games.get('available') and games.get('score_distribution', {}).get('bin_centers'):
    histogram = pd.DataFrame({'total points (bin center)': games['score_distribution']['bin_centers'],
                              'games': games['score_distribution']['counts']})
    px.bar(histogram, x='total points (bin center)', y='games', title='Game total-score distribution').show()
else:
    print('Game score distribution unavailable:', games.get('reason', games.get('unavailable_analyses')))